# An Introduction to Static Typing in OCaml

The Python version of this notebook introduces *gradual typing*: Python programs may contain *type
annotations*, but the Python interpreter ignores them.  Only a separate tool, a *type checker* like
[mypy](https://mypy-lang.org/) or [Pyright](https://github.com/microsoft/pyright), uses them to find errors.

OCaml is different: it is a *statically typed* language.  Every expression has a type that is known
before the program runs, and a program that contains a type error is not executed at all.  Nevertheless, OCaml
programs need very few type annotations, since the compiler *infers* the types of variables, parameters,
and functions.  The type system of OCaml is very expressive.  Among other things, it offers
+ *generic types* (called *polymorphic types* in OCaml),
+ *variant types*, which correspond to the *union types* of Python, and
+ *user defined types* like records and recursive types.

This notebook shows the OCaml counterparts of the examples of the Python notebook.  As in Python, type
annotations make the code more self-explanatory.  In OCaml, they are checked by the compiler every time.

## The Type Checker

In OCaml, the type checker is part of the compiler, and the OCaml kernel of *Jupyter* type checks every
cell before it is executed.  If a cell contains a type error, the error is reported with the line and the
columns of the offending expression, and the cell is not executed.  Note that this notebook contains some
errors on purpose, in order to show how the type checker finds them.  These cells are tagged with
`raises-exception`, so that the notebook can be executed in one go.

After a definition has been executed, the notebook shows the type that the compiler has inferred, e.g.
`val add : int -> int -> int = <fun>`.

## Finding Errors by Type Checking

The following two lines contain an error that the type checker is able to find: the function `input` returns a string and a string cannot be added to a number.

The OCaml kernel of *Jupyter* can not read from the keyboard: the function `read_line` raises the exception
`End_of_file`.  Therefore, the next cell defines a function `input` that pretends that the user has typed `42`.

In [ ]:
#use "../lib/nb.ml";;

(* The OCaml kernel can not read from the keyboard, so we pretend that the user has typed 42. *)
let input (prompt : string) : string =
  print_endline (prompt ^ "42");
  "42"

In [ ]:
let number = input "What is your favourite number? " in
Printf.printf "It is %d\n%!" (number + 1)

The correct version of the two lines above would have been as follows.  The function `int_of_string`
converts the string into a number.

In [ ]:
let number = int_of_string (input "What is your favourite number? ") in
Printf.printf "It is %d.\n%!" (number + 1)

## Type Annotations

In OCaml, the type of a parameter is annotated by enclosing the parameter in parentheses and putting a colon
after its name.  The return type of a function is specified by a colon after the parameters, as shown below.
All of these annotations are optional.

**Function `add a b`**
- *Input:* `a` and `b` are integers.
- *Output:* The sum of `a` and `b`.

In [ ]:
let add (a : int) (b : int) : int =
  a + b

In the next cell the **type checker** tells us, that we have called the function `add` with strings instead of integers.
In Python, the **interpreter** executes such a cell without encountering an error, since the interpreter does not
care about the type annotations.  In OCaml, the call of `add` is rejected *before* it is executed.  (The
first line of the cell is executed, since the cell consists of two separate definitions.)

In [ ]:
let name = "Karl";;
add "Hello " name

In Python, the type annotations of a function can be inspected at runtime via the attribute `__annotations__`.
In OCaml, types only exist at compile time.  However, the notebook shows the type of every expression, so
we can look at the type of a function by evaluating its name.  The type `int -> int -> int` is the type of a
function that takes two integers and returns an integer.

In [ ]:
add

If we do not use type annotations for a function `f`, the type of `f` is *inferred* by the compiler.  Since the
operator `-` can only be applied to integers (floating point numbers are subtracted with `-.`), the compiler
knows that `sub` takes two integers and returns an integer.

**Function `sub a b`**
- *Input:* `a` and `b` are numbers.  This function has no type annotations.
- *Output:* The difference `a - b`.

In [ ]:
let sub a b =
  a - b

In [ ]:
sub

## Built-in Types

OCaml supports basic types like `int`, `float`, `string`, `char`, and `bool`.
Complex types like lists (`int list`), tuples (`int * string`), arrays (`float array`), and hash tables
(`(string, int) Hashtbl.t`) are also supported.  Note that all elements of a list have the same type.

The function `average numbers` computes the arithmetic mean of the numbers in the list `numbers`.

In Python, the type `int|float` admits both integers and floating point numbers.  In OCaml, integers and
floating point numbers are different types and there is no implicit conversion between them.  Therefore, the
function `average` takes a list of floating point numbers.  The function `List.fold_left (+.) 0. numbers` adds up
all elements of `numbers`, and `float_of_int` converts the length of the list into a floating point number.

**Function `average numbers`**
- *Input:* `numbers` is a non-empty list of numbers.
- *Output:* The arithmetic mean of `numbers`.

In [ ]:
let average (numbers : float list) : float =
  List.fold_left (+.) 0. numbers /. float_of_int (List.length numbers)

In [ ]:
average [1.; 2.; 3.; 4.]

## Custom Types
In Python, you define your own types using the keyword `class`.  OCaml has classes, too, but most of the
time, a *record type* together with some functions is simpler.  The record type `person` has a single *field*
`name`.  Instead of the method `greet`, we define a function `greet` that takes a person as its argument.

In [ ]:
type person = { name : string }

let greet (p : person) : string =
  Printf.sprintf "Hello, %s!" p.name

For comparison, the cell below shows the class `person_class`, which is the direct counterpart of the
Python class `Person`.  The constructor arguments follow the name of the class, and the method `greet` is
called as `p#greet`.

In [ ]:
class person_class (name : string) =
  object
    method greet : string = Printf.sprintf "Hello, %s!" name
  end
;;
(new person_class "Brutus")#greet

When a function does not return a value, the return type is `unit`.  The only value of this type is `()`,
which corresponds to Python's `None`.

**Function `salve p`**
- *Input:* `p` is a person.
- *Output:* `()`.  The function prints a greeting for `p`.

In [ ]:
let salve (p : person) : unit =
  print_endline (greet p)

In [ ]:
let jc = { name = "Julius Caesar" };;
salve jc

The function `greet_name` either accepts a string representing a name as its argument,
or it accepts a given name together with a family name.  In Python, the latter is a dictionary that stores
the first name under the key `given` and the last name under the key `family`.

In Python, the *union* operator `|` is used to express the fact that `name` can either be a
`str` or a `dict[str, str]`.  In OCaml, we define a *variant type* instead: a value of type `name` is either
`Simple s`, where `s` is a string, or `Full { given; family }`, where `given` and `family` are strings.
A function that uses `match` on a value of a variant type has to handle all cases, and the compiler checks
this.  Moreover, the compiler ensures that a full name always contains both a given name and a family name,
while a Python dictionary might lack the key `family`.

**Function `greet_name name`**
- *Input:* `name` is either a simple name or a full name consisting of a given name and a family name.
- *Output:* A greeting for the given name.

In [ ]:
type name =
  | Simple of string
  | Full of { given : string; family : string }

let greet_name (name : name) : string =
  match name with
  | Simple s -> "Hi " ^ s ^ "!"
  | Full { given; family } -> Printf.sprintf "Bienvenido, Señor %s %s." given family

In [ ]:
greet_name (Simple "Alice")

In [ ]:
greet_name (Full { given = "Esteban"; family = "Ramirez" })

## Generic Functions and Type Variables

The next example shows how to type *generic* functions.  In Python, type variables have to be created with the
function `TypeVar`.  In OCaml, *type variables* are written with a leading apostrophe, e.g. `'s` and `'t`.
They do not have to be declared.  Generic functions are called *polymorphic* functions in OCaml.

The function `swap` takes a pair (a 2-tuple) and swaps its components.  The first component has the type `'s`,
the second component has the type `'t`.  The function `swap` is *polymorphic*, meaning it is able to handle pairs
of integers, strings, or any other types.  The type `'s * 't` is the type of pairs whose first component has the
type `'s` and whose second component has the type `'t`.  The pattern `(x, y)` extracts the components.

**Function `swap pair`**
- *Input:* `pair` is a pair whose components have the types `'s` and `'t`.
- *Output:* The pair with its two components swapped.

In [ ]:
let swap ((x, y) : 's * 't) : 't * 's =
  (y, x)

In the next cell, both type variables `'s` and `'t` are instantiated as `int`.

In [ ]:
swap (1, 2)

In the following cell, both type variables `'s` and `'t` are instantiated as `string`.

In [ ]:
swap ("a", "b")

Below, the type variable `'s` is instantiated as `int`, while `'t` is instantiated as `string`.

In [ ]:
swap (1, "a")

## Explicitly Polymorphic Type Annotations

Python 3.12 allows to list the type variables of a generic function in square brackets.  The OCaml counterpart is an
*explicitly polymorphic* type annotation: `'u 'v 'w. 'u * 'v * 'w -> 'v * 'w * 'u` states that the function
works for *all* types `'u`, `'v`, and `'w`.  The compiler reports an error if the function is less general than
its annotation, e.g. if it only works for integers.

**Function `rotate triple`**
- *Input:* `triple` is a triple whose components have the types `'u`, `'v`, and `'w`.
- *Output:* The triple rotated to the left by one position.

In [ ]:
let rotate : 'u 'v 'w. 'u * 'v * 'w -> 'v * 'w * 'u =
  fun (x, y, z) -> (y, z, x)

In [ ]:
rotate (1, "a", true)

## Recursive Types

In Python, the type `Expression` is defined as a union of the types `float`, `str`, and a type of tuples, and the
type `Operator` is a set of string literals.  In OCaml, both are *variant types*.  The type `expression` is
*recursive*, since the constructor `BinOp` contains two expressions.

In [ ]:
type operator = Plus | Minus | Times | Divide

In [ ]:
type expression =
  | Const of float
  | Var   of string
  | BinOp of expression * operator * expression

The function `differentiate ~var e` takes an arithmetic expression `e` and a variable `var`
and computes the derivative of `e` with respect to `var`.  Since `var` has the default value `"x"`, it is an
*optional labelled argument*, which is written as `?(var = "x")` and passed as `~var:"y"`.

**Function `differentiate ~var expr`**
- *Input:* `expr` is an arithmetic expression and `var` is the name of a variable.
- *Output:* The derivative of `expr` with respect to `var`.

In [ ]:
let rec differentiate ?(var : string = "x") (expr : expression) : expression =
  match expr with
  | Const _ -> Const 0.0
  | Var x -> if x = var then Const 1.0 else Const 0.0
  | BinOp (lhs, op, rhs) ->
      let dLhs : expression = differentiate ~var lhs in
      let dRhs : expression = differentiate ~var rhs in
      match op with
      | Plus  -> BinOp (dLhs, Plus, dRhs)
      | Minus -> BinOp (dLhs, Minus, dRhs)
      | Times ->
          let p1 : expression = BinOp (dLhs, Times, rhs) in
          let p2 : expression = BinOp (lhs, Times, dRhs) in
          BinOp (p1, Plus, p2)
      | Divide ->
          let q1 : expression = BinOp (dLhs, Times, rhs) in
          let q2 : expression = BinOp (lhs, Times, dRhs) in
          let numerator   : expression = BinOp (q1, Minus, q2) in
          let denominator : expression = BinOp (rhs, Times, rhs) in
          BinOp (numerator, Divide, denominator)

The following version of `differentiate` does not declare the types of its local variables.  In Python, the type
checker reports an error in the case of a division: it infers the type of the variable `numerator` from the
tuple on its right-hand side, and in doing so it widens the operators `'*'` and `'-'` to the type `str`.  This is
the reason why the Python version needs type annotations for its local variables.

In OCaml, this problem does not occur: `Times` and `Minus` are constructors of the type `operator`, so the
inferred type of `numerator` is `expression`.  Both versions are accepted.  Note also that the Python version needs
the final line `return 0.0`, since the type checker can not see that the cases of `match` are complete.  In
OCaml, the compiler checks that every case is handled, so no such line is needed.

**Function `differentiate(expr, var)`**
- *Input:* `expr` is an arithmetic expression and `var` is the name of a variable.
- *Output:* The derivative of `expr` with respect to `var`.

In [ ]:
let rec differentiate ?(var : string = "x") (expr : expression) : expression =
  match expr with
  | Const _ -> Const 0.0
  | Var x -> if x = var then Const 1.0 else Const 0.0
  | BinOp (lhs, op, rhs) ->
      let dLhs = differentiate ~var lhs in
      let dRhs = differentiate ~var rhs in
      match op with
      | Plus   -> BinOp (dLhs, Plus, dRhs)
      | Minus  -> BinOp (dLhs, Minus, dRhs)
      | Times  -> BinOp (BinOp (dLhs, Times, rhs), Plus, BinOp (lhs, Times, dRhs))
      | Divide ->
          let numerator   = BinOp (BinOp (dLhs, Times, rhs), Minus, BinOp (lhs, Times, dRhs)) in
          let denominator = BinOp (rhs, Times, rhs) in
          BinOp (numerator, Divide, denominator)

In [ ]:
differentiate ~var:"x" (BinOp (Const 1.0, Divide, BinOp (Const 1.0, Plus, BinOp (Var "x", Times, Var "x"))))